# **Environment Setup & Imports**

In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")

TensorFlow Version: 2.19.0
Keras Version: 3.10.0


# **DeepLabV3+ Model Architecture**

In [3]:
def ConvolutionBlock(block_input, num_filters, kernel_size=3, dilation_rate=1, padding="same"):
    x = layers.Conv2D(num_filters, kernel_size=kernel_size, dilation_rate=dilation_rate, padding=padding, use_bias=False)(block_input)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x) # Fixed: Using Keras layer instead of tf.nn

def DilatedSpatialPyramidPooling(dspp_input):
    dims = dspp_input.shape
    # Standard 1x1 conv
    out_1 = ConvolutionBlock(dspp_input, 256, kernel_size=1)
    # Dilated convs for different scales
    out_6 = ConvolutionBlock(dspp_input, 256, dilation_rate=6)
    out_12 = ConvolutionBlock(dspp_input, 256, dilation_rate=12)
    out_18 = ConvolutionBlock(dspp_input, 256, dilation_rate=18)
    
    # Global average pooling
    image_features = layers.GlobalAveragePooling2D()(dspp_input)
    image_features = layers.Reshape((1, 1, dims[-1]))(image_features)
    image_features = ConvolutionBlock(image_features, 256, kernel_size=1)
    image_features = layers.UpSampling2D(size=(dims[1], dims[2]), interpolation="bilinear")(image_features)

    x = layers.Concatenate(axis=-1)([image_features, out_1, out_6, out_12, out_18])
    return ConvolutionBlock(x, 256, kernel_size=1)

def DeeplabV3Plus(image_size, num_classes):
    model_input = keras.Input(shape=(image_size, image_size, 3))
    
    # Pre-trained MobileNetV2 Backbone
    backbone = keras.applications.MobileNetV2(weights="imagenet", include_top=False, input_tensor=model_input)
    
    # Encoder part
    x = backbone.get_layer("block_13_expand_relu").output 
    x = DilatedSpatialPyramidPooling(x)
    
    # Decoder part
    input_a = layers.UpSampling2D(size=(4, 4), interpolation="bilinear")(x)
    input_b = backbone.get_layer("block_3_expand_relu").output
    input_b = ConvolutionBlock(input_b, 48, kernel_size=1)

    x = layers.Concatenate(axis=-1)([input_a, input_b])
    x = ConvolutionBlock(x, 256)
    x = ConvolutionBlock(x, 256)
    x = layers.UpSampling2D(size=(4, 4), interpolation="bilinear")(x)
    
    # Output layer
    model_output = layers.Conv2D(num_classes, kernel_size=(1, 1), padding="same")(x)
    return keras.Model(inputs=model_input, outputs=model_output)

# Create the model
IMAGE_SIZE = 512
NUM_CLASSES = 3 # 0:Background, 1:Wall, 2:Floor
model = DeeplabV3Plus(image_size=IMAGE_SIZE, num_classes=NUM_CLASSES)
model.summary()

/tmp/ipykernel_57/3142773362.py:28: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  backbone = keras.applications.MobileNetV2(weights="imagenet", include_top=False, input_tensor=model_input)
I0000 00:00:1778770160.003079      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778770160.008986      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 512, 512,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 256, 256,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 256, 256,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 256, 256,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 256, 256,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 256, 256,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 256, 256,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 256, 256,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 256, 256,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 256, 256,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 256, 256,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 256, 256,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 257, 257,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 128, 128,  │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 128, 128,  │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 128, 128,  │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 128, 128,  │      2,304 │ block_1_depthwis

 Total params: 6,526,467 (24.90 MB)

 Trainable params: 6,504,995 (24.81 MB)

 Non-trainable params: 21,472 (83.88 KB)

# **Training Configuration (Loss & Optimizer)**

In [4]:
# We use SparseCategoricalCrossentropy because our labels are integers (0, 1, 2)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001), 
    loss=loss_fn, 
    metrics=["accuracy"]
)

print("✅ Training Configuration Complete.")

✅ Training Configuration Complete.


# **Accuracy Evaluation (mIoU)**

In [5]:
# Mean Intersection over Union (mIoU) is the best metric for Segmentation
miou_metric = keras.metrics.MeanIoU(num_classes=NUM_CLASSES)

print("✅ Accuracy Metric (mIoU) Initialized.")

✅ Accuracy Metric (mIoU) Initialized.
